# HDFC Bank: Fraud Risk Analysis - Part 2b

**Objective:** Rigorous Feature Selection

Instead of arbitrarily picking 12 features, a real ML engineer uses a scientific funnel to extract the most predictive columns from the 400+ raw columns.

We will use our custom `RigorousFeatureSelector` which applies:
1. **Missingness Filter:** Drop columns > 90% missing.
2. **Zero-Variance Filter:** Drop constant columns.
3. **Tree-Based Multivariate Selection:** Train a Random Forest and extract the Top 50 most predictive features.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

from fraudguard.data.ingestion import load_bank_data, split_temporal
from fraudguard.features.selection import RigorousFeatureSelector

# 1. Load Data
data_dir = Path.cwd().parent / "data" / "raw"
df = load_bank_data(data_dir)

# 2. Split Temporally
# We MUST fit our feature selector ONLY on the training data!
df_train, df_test = split_temporal(df, test_ratio=0.2)
X_train = df_train.drop(columns=['isFraud'])
y_train = df_train['isFraud'].values

## 1. Running the Feature Selection Funnel
Let's only look at numeric columns for this demonstration (since tree models require numbers). We will pass the hundreds of `V`, `C`, and `D` features through the funnel.

In [2]:
# Extract numeric columns only for the selector
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
X_train_numeric = X_train[numeric_cols]

print(f"Starting with {X_train_numeric.shape[1]} raw numeric features...")

# Initialize our custom selector to keep the Top 50 features
selector = RigorousFeatureSelector(max_missing_ratio=0.90, top_n_features=50)

# Fit the selector (this will print out the funnel progression)
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

selector.fit(X_train_numeric, y_train)

Starting feature selection funnel. Initial features: 402


Starting with 402 raw numeric features...


Features after Missingness Filter (<90.0%): 392
Features after Zero-Variance Filter: 392
Training shallow Random Forest to extract multivariate feature importance...
Selected Top 50 highly predictive features.


,max_missing_ratio,0.9
,top_n_features,50
,random_state,42
Name,Type,Value
selected_features_,list,"['V200', 'V189', 'V258', 'V246', ...]"


## 2. The Winning Features
Let's see which features mathematically proved themselves to be the most valuable for catching fraud!

In [3]:
print("\nTop 50 Selected Features:")
print(selector.selected_features_)

# Transform the data (drops the other 350+ useless features)
X_train_reduced = selector.transform(X_train_numeric)
print(f"\nFinal Reduced Matrix Shape: {X_train_reduced.shape}")


Top 50 Selected Features:
['V200', 'V189', 'V258', 'V246', 'V188', 'V257', 'C1', 'V201', 'V199', 'C12', 'C4', 'V44', 'V243', 'V187', 'V45', 'V244', 'V231', 'C7', 'V245', 'C8', 'V242', 'V232', 'C13', 'V259', 'V154', 'V198', 'C6', 'V275', 'V156', 'C10', 'V149', 'V87', 'V318', 'V295', 'V230', 'V86', 'C2', 'C14', 'V229', 'V177', 'V254', 'C11', 'V194', 'V134', 'V210', 'V317', 'V294', 'id_17', 'V172', 'V186']

Final Reduced Matrix Shape: (472432, 50)


In [4]:
# Extract numeric columns only for the selector
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
X_train_numeric = X_train[numeric_cols]

print(f"Starting with {X_train_numeric.shape[1]} raw numeric features...")

# Initialize our custom selector to keep the Top 50 features
selector = RigorousFeatureSelector(max_missing_ratio=0.90, top_n_features=12)

# Fit the selector (this will print out the funnel progression)
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

selector.fit(X_train_numeric, y_train)

Starting feature selection funnel. Initial features: 402


Starting with 402 raw numeric features...


Features after Missingness Filter (<90.0%): 392
Features after Zero-Variance Filter: 392
Training shallow Random Forest to extract multivariate feature importance...
Selected Top 12 highly predictive features.


,top_n_features,12
,max_missing_ratio,0.9
,random_state,42
Name,Type,Value
selected_features_,list,"['V200', 'V189', 'V258', 'V246', ...]"


In [6]:
print("\nTop 15 Selected Features:")
print(selector.selected_features_)

# Transform the data (drops the other 350+ useless features)
X_train_reduced = selector.transform(X_train_numeric)
print(f"\nFinal Reduced Matrix Shape: {X_train_reduced.shape}")


Top 15 Selected Features:
['V200', 'V189', 'V258', 'V246', 'V188', 'V257', 'C1', 'V201', 'V199', 'C12', 'C4', 'V44']

Final Reduced Matrix Shape: (472432, 12)
